In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


base_dir = "path/to/project/"
cha_pan = os.path.join(base_dir, "final_analysis/data/CHA/pan/bp35w60")

chroms = [str(i) for i in range(1,23)]

maps_pan = {}
chrom_sizes = {}

for chrom in chroms:
    p = os.path.join(cha_pan, f"CHA_recombmap_chr{chrom}_bp35w60")
    dfm = pd.read_csv(p, sep="\t", header=None, names=["Start","End","Rec.Rate"])
    maps_pan[chrom] = dfm
    chrom_sizes[chrom] = int(dfm["End"].max())

print("Loaded CHA recomb maps")

In [ ]:
segment_duplication_dir = os.path.join(
    base_dir,
    "final_analysis/data/segment_duplication/mafft.tsv"
)

sd = pd.read_csv(segment_duplication_dir, sep="\t", header=None,names=["GID","chrA","sA","eA","chrB","sB","eB", "strand","score","L","mism","gap"])

In [ ]:
sd["valid"] = sd["L"] - sd["gap"]
sd["pdist"] = sd["mism"] / sd["valid"]
print(sd['pdist'].describe())

In [ ]:
p = sd["pdist"].astype(float)

sd["jc"] = np.where(
    np.isfinite(p) & (p >= 0) & (p < 0.75),
    -0.75 * np.log(1 - (4/3)*p),
    np.nan
)

In [ ]:
def clip_map_to_allowed_regions(df_map: pd.DataFrame,
                               allowed_df: pd.DataFrame,
                               chrom: str,
                               chrom_col_allowed: str = "chr",
                               start_col_allowed: str = "Start",
                               end_col_allowed: str = "End") -> pd.DataFrame:
    """
    Keep only parts of df_map that overlap allowed intervals for `chrom`.
    If a row crosses an allowed boundary, trim it (may split into multiple rows).
    Assumes df_map has columns: Start, End, Rec.Rate (bp coordinates, half-open-ish).
    """
    # Allowed intervals for this chromosome
    allowed = allowed_df[allowed_df[chrom_col_allowed].astype(str) == str(chrom)][
        [start_col_allowed, end_col_allowed]
    ].copy()

    if allowed.empty or df_map.empty:
        return df_map.iloc[0:0].copy()

    # sort for safety
    allowed = allowed.sort_values([start_col_allowed, end_col_allowed]).to_numpy()
    df_map = df_map.sort_values(["Start", "End"]).reset_index(drop=True)

    out_rows = []

    # Two-pointer sweep (fast enough; map windows are usually not huge)
    j = 0
    for s, e, r in df_map[["Start", "End", "Rec.Rate"]].to_numpy():
        if e <= s:
            continue

        # advance allowed pointer until it might overlap
        while j < len(allowed) and allowed[j][1] <= s:
            j += 1

        k = j
        # collect all overlaps with allowed intervals
        while k < len(allowed) and allowed[k][0] < e:
            a_s, a_e = allowed[k]
            ov_s = max(s, a_s)
            ov_e = min(e, a_e)
            if ov_s < ov_e:
                out_rows.append((ov_s, ov_e, r))
            if a_e >= e:
                break
            k += 1

    if not out_rows:
        return df_map.iloc[0:0].copy()

    df_out = pd.DataFrame(out_rows, columns=["Start", "End", "Rec.Rate"])
    df_out = df_out.sort_values(["Start", "End"]).reset_index(drop=True)
    return df_out

In [ ]:
# remove unavailable region

pan_available = "CHM13v2.telo_cent.complement.bed"

pan_available_region = pd.read_csv(
	pan_available, sep="\t", header=None, names=["Chrom", "Start", "End"]
)
# remove chrX and chrY in Chrom
pan_available_region = pan_available_region[~pan_available_region["Chrom"].isin(["chrX", "chrY"])]
pan_available_region ["chr"] = pan_available_region ["Chrom"].str.replace("chr", "")
pan_available_region

In [ ]:
for chrom in chroms:
	maps_pan[str(chrom)] = clip_map_to_allowed_regions(
        df_map=maps_pan[str(chrom)],
        allowed_df=pan_available_region,
        chrom=str(chrom),          # your chroms is like "1","2",...
        chrom_col_allowed="chr",   # column in pan_available_region
        start_col_allowed="Start",
        end_col_allowed="End"
    )

In [ ]:
sd["chrA_n"] = sd["chrA"].str.replace("^chr", "", regex=True)
sd["chrB_n"] = sd["chrB"].str.replace("^chr", "", regex=True)


def weighted_mean_rate(map_df, start, end):
    """Weighted mean Rec.Rate over [start, end) using overlap length as weights."""
    ov = map_df[(map_df["End"] > start) & (map_df["Start"] < end)].copy()
    if ov.empty:
        return np.nan
    ov["Start"] = np.maximum(ov["Start"].to_numpy(), start)
    ov["End"]   = np.minimum(ov["End"].to_numpy(),   end)
    w = (ov["End"] - ov["Start"]).to_numpy()
    # safety: if all weights are 0, return NaN
    if w.sum() == 0:
        return np.nan
    return np.average(ov["Rec.Rate"].to_numpy(), weights=w)

# --- compute mean recombination for each SD copy ---
meanA_list = []
meanB_list = []

for _, r in sd.iterrows():
    chromA = r["chrA_n"]
    chromB = r["chrB_n"]

    # NOTE: maps_pan keys must match chrom strings (e.g. "chr10")
    mA = maps_pan[chromA]
    mB = maps_pan[chromB]

    if mA is None or mB is None:
        meanA_list.append(np.nan)
        meanB_list.append(np.nan)
        continue

    sA, eA = int(r["sA"]), int(r["eA"])
    sB, eB = int(r["sB"]), int(r["eB"])

    meanA_list.append(weighted_mean_rate(mA, sA, eA))
    meanB_list.append(weighted_mean_rate(mB, sB, eB))

sd["meanA"] = meanA_list
sd["meanB"] = meanB_list
sd["abs_diff_mean"] = (sd["meanA"] - sd["meanB"]).abs()
sd["abs_log_diff"] =(np.log10(sd["meanA"]) - np.log10(sd["meanB"])).abs()

# Optional but often useful for “pair recombination environment”
sd["mean_pair"] = sd[["meanA", "meanB"]].mean(axis=1)
sd["max_pair"]  = sd[["meanA", "meanB"]].max(axis=1)
sd["min_pair"]  = sd[["meanA", "meanB"]].min(axis=1)


print("Total SD rows:", len(sd))
print("Rows with meanA available:", sd["meanA"].notna().sum())
print("Rows with meanB available:", sd["meanB"].notna().sum())

In [ ]:
# only keep sd rows when both meanA and meanB are not NaN
sd_valid = sd.dropna(subset=["meanA", "meanB"]).copy()
sd_valid

In [ ]:

import pingouin as pg


In [ ]:
# filter rows with jc > 0.1
sd_filtered = sd_valid[sd_valid["jc"] <= 0.1].copy()
sd_filtered
sd_valid_divergence = sd_filtered[sd_filtered["pdist"] <= 0.1].copy()

pcorr = pg.partial_corr(
    data=sd_valid_divergence,
    x="meanA",
    y="meanB",
    covar="L",
    method="spearman"
)

print(pcorr)

In [ ]:
sd_valid_divergence

# write to csv of sd_valid_divergence
sd_valid_divergence.to_csv("sd_analysis_final.csv")


In [ ]:
# do the spearman correlation separately for inter- and intra-chromosomal SDs

sd_valid_divergence["pair_type"] = np.where(
    sd_valid_divergence["chrA_n"] != sd_valid_divergence["chrB_n"],
    "Interchromosomal",
    "Intrachromosomal"
)


for pair_type in ["Intrachromosomal", "Interchromosomal"]:
	subset = sd_valid_divergence[sd_valid_divergence["pair_type"] == pair_type]
	pcorr = pg.partial_corr(
		data=subset,
		x="meanA",
		y="meanB",
		covar="L",
		method="spearman"
	)
	print(f"=== {pair_type} ===")
	print(pcorr)

In [ ]:

plt.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial"],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "font.size": 11,
        "axes.linewidth": 1.2
    })
plt.figure(figsize=(10, 8))

colors = {
    "Intrachromosomal": "#1f77b4",  # blue
    "Interchromosomal": "#d62728"   # red
}

for pair_type, color in colors.items():
    subset = sd_valid_divergence[sd_valid_divergence["pair_type"] == pair_type]
    plt.scatter(
        subset["meanA"],
        subset["meanB"],
        label=pair_type,
        alpha=0.4,
        color=color
    )

plt.xscale("log")
plt.yscale("log")
plt.xlim(1e-11, 1e-7)
plt.ylim(1e-11, 1e-7)




# y = x reference
plt.plot(
    [1e-11, 1e-7],
    [1e-11, 1e-7],
    color="black",
    linestyle="--",
    linewidth=1,
    label="y = x"
)



plt.xlabel("Mean Recombination Rate of Copy A")
plt.ylabel("Mean Recombination Rate of Copy B")
plt.title("Mean Recombination Rates of a SD Pair")
plt.legend()
plt.grid(True, which="both", ls="--", linewidth=0.5)
plt.savefig("sd_recombination_rate_comparison.pdf", bbox_inches="tight")

